# CardioIA Fase 4: a diferença entre os dois modelos é real?

**Autor:** Silvio Prestes Guerreiro Junior, RM 567958. Protótipo acadêmico, sem validade diagnóstica.

O conjunto de teste tem 819 imagens. Com esse tamanho, diferenças de um ou dois pontos percentuais podem ser fruto do acaso da amostra. Este notebook curto usa as previsões gravadas pelos notebooks `02` e `03` (nenhum modelo é treinado aqui) para medir a incerteza das métricas:

- **intervalo de confiança de 95% por bootstrap**, reamostrando **pacientes** com reposição (2.000 repetições), já que imagens do mesmo paciente não são independentes;
- **diferença pareada** entre os modelos, nas mesmas reamostras;
- **teste de McNemar** sobre os acertos e erros dos dois modelos nas mesmas imagens.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
from scipy.stats import binomtest

SEED = 42
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
man = pd.read_csv(RAIZ / "data" / "manifesto.csv")
a = pd.read_csv(RAIZ / "data" / "pred_teste_cnn_do_zero.csv").rename(columns={"prob": "p_cnn"})
b = pd.read_csv(RAIZ / "data" / "pred_teste_resnet50.csv").rename(columns={"prob": "p_res"})
d = a.merge(b[["arquivo", "p_res"]], on="arquivo").merge(man[["arquivo", "paciente"]], on="arquivo")
assert len(d) == len(a) == len(b)
print(len(d), "imagens |", d.paciente.nunique(), "pacientes")

def metricas(y, p):
    yp = (p >= 0.5).astype(int)
    return {"acuracia": accuracy_score(y, yp), "precisao": precision_score(y, yp), "recall": recall_score(y, yp),
            "f1": f1_score(y, yp), "auc": roc_auc_score(y, p)}

rng = np.random.default_rng(SEED)
grupos = {k: g.index.values for k, g in d.groupby("paciente")}
pacs = np.array(list(grupos))
linhas = []
for _ in range(2000):
    idx = np.concatenate([grupos[k] for k in rng.choice(pacs, len(pacs), replace=True)])
    s = d.loc[idx]
    mc, mr = metricas(s.y.values, s.p_cnn.values), metricas(s.y.values, s.p_res.values)
    linhas.append({**{f"cnn_{k}": v for k, v in mc.items()}, **{f"res_{k}": v for k, v in mr.items()},
                   **{f"dif_{k}": mr[k] - mc[k] for k in mc}})
bs = pd.DataFrame(linhas)

819 imagens | 652 pacientes


In [2]:
pc, pr = metricas(d.y.values, d.p_cnn.values), metricas(d.y.values, d.p_res.values)
tab = []
for k in pc:
    ic = lambda col: f"[{bs[col].quantile(.025):.3f}, {bs[col].quantile(.975):.3f}]"
    lo, hi = bs[f"dif_{k}"].quantile(.025), bs[f"dif_{k}"].quantile(.975)
    tab.append({"metrica": k, "CNN do zero": f"{pc[k]:.3f} {ic('cnn_' + k)}", "ResNet50": f"{pr[k]:.3f} {ic('res_' + k)}",
                "diferenca (ResNet50 - CNN)": f"{pr[k] - pc[k]:+.3f} [{lo:+.3f}, {hi:+.3f}]",
                "IC exclui zero": "sim" if (lo > 0 or hi < 0) else "não"})
tab = pd.DataFrame(tab).set_index("metrica")
tab.to_csv(RAIZ / "data" / "comparacao_bootstrap.csv")
tab

,CNN do zero,ResNet50,diferenca (ResNet50 - CNN),IC exclui zero
metrica,,,,
acuracia,"0.799 [0.768, 0.829]","0.807 [0.776, 0.836]","+0.009 [-0.019, +0.037]",não
precisao,"0.787 [0.734, 0.833]","0.770 [0.717, 0.814]","-0.017 [-0.049, +0.013]",não
recall,"0.808 [0.758, 0.851]","0.866 [0.825, 0.901]","+0.057 [+0.020, +0.099]",sim
f1,"0.798 [0.754, 0.834]","0.815 [0.774, 0.848]","+0.017 [-0.010, +0.046]",não
auc,"0.874 [0.845, 0.900]","0.882 [0.855, 0.906]","+0.007 [-0.012, +0.027]",não


In [3]:
# McNemar: imagens em que só um dos modelos acerta (limiar 0,5)
ac, ar = (d.p_cnn >= 0.5).astype(int) == d.y, (d.p_res >= 0.5).astype(int) == d.y
so_res, so_cnn = int((ar & ~ac).sum()), int((ac & ~ar).sum())
p = binomtest(so_res, so_res + so_cnn, 0.5).pvalue
print(f"Ambos acertam: {int((ac & ar).sum())} | ambos erram: {int((~ac & ~ar).sum())} | só a ResNet50 acerta: {so_res} | só a CNN acerta: {so_cnn}")
print(f"McNemar exato (acurácia): p = {p:.3f}")

# o mesmo, restrito aos casos reais de cardiomegalia (diferença de recall)
m = d.y == 1
s_res, s_cnn = int((ar & ~ac & m).sum()), int((ac & ~ar & m).sum())
print(f"Casos positivos: só a ResNet50 detecta {s_res} | só a CNN detecta {s_cnn} | McNemar exato (recall): p = {binomtest(s_res, s_res + s_cnn, 0.5).pvalue:.4f}")
m = d.y == 0
s_res, s_cnn = int((ar & ~ac & m).sum()), int((ac & ~ar & m).sum())
print(f"Casos negativos: só a ResNet50 acerta {s_res} | só a CNN acerta {s_cnn} | McNemar exato (especificidade): p = {binomtest(s_res, s_res + s_cnn, 0.5).pvalue:.4f}")
print(f"Correlação entre as probabilidades dos dois modelos: {np.corrcoef(d.p_cnn, d.p_res)[0, 1]:.3f}")

Ambos acertam: 589 | ambos erram: 93 | só a ResNet50 acerta: 72 | só a CNN acerta: 65
McNemar exato (acurácia): p = 0.608
Casos positivos: só a ResNet50 detecta 43 | só a CNN detecta 20 | McNemar exato (recall): p = 0.0052
Casos negativos: só a ResNet50 acerta 29 | só a CNN acerta 45 | McNemar exato (especificidade): p = 0.0805
Correlação entre as probabilidades dos dois modelos: 0.815


## Leitura

- **Acurácia, precisão, F1 e AUC:** os intervalos de confiança dos dois modelos se sobrepõem quase por completo e o intervalo da diferença inclui o zero nas quatro métricas. O teste de McNemar para a acurácia deu p = 0,61. Não há evidência de que um modelo acerte mais que o outro.
- **Recall no limiar 0,5:** a ResNet50 detecta mais casos de Cardiomegalia (+5,7 pontos, intervalo de +2,0 a +9,9; McNemar p = 0,005). Nos casos negativos a tendência é inversa, a favor da CNN (29 contra 45, p = 0,08). Juntos, os dois resultados indicam que, no limiar 0,5, a ResNet50 está em um ponto de operação mais sensível e menos específico, e não que ela separe melhor as classes. O AUC, que não depende do limiar, confirma: a diferença é de 0,007, com intervalo de -0,012 a +0,027.
- **Os modelos erram em imagens diferentes:** em 137 das 819 imagens só um deles acerta, e a correlação entre as probabilidades é de 0,815. Combinar os dois (média das probabilidades) é uma melhoria possível, não explorada aqui.
- **Precisão das estimativas:** com 819 imagens, cada métrica tem margem de cerca de 3 a 5 pontos percentuais para cada lado. Qualquer comparação entre os modelos deve ser lida com essa margem.